# 01 · Data Cleaning & Standardization (2015–2025)

**Project:** AgriShield — Early Warning System for Crop Yield & Livestock Forage Risk in Kenya  
**Notebook Goal:** Clean our raw datasets from `../data/`, standardize all **47 Kenyan county names**, align the time series to **monthly county records (`2015–2025`, `6,204` rows)**, and combine them into a unified dataset in `../data/processed/` for EDA, Feature Engineering, and Model Training.

---
### Step 1: Setup, Output Folder & Standard 47-County Name Helper
Different raw files spell county names differently (for example: `KE028`, `ELGEYO/MARAKWET`, `Elgeyo-Marakwet`, or `MURANG'A`).  
In the cell below, we:
1. Create the `../data/processed` folder where cleaned files will be saved.
2. Load `kenya_county_crosswalk.csv` and build a simple `clean_county_name()` function so every dataset uses the exact same **47 standard county names**.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# 1. Set up input (raw) and output (processed) directories

RAW_DIR = Path("../data")
PROC_DIR = Path("../data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

# 2. Load the crosswalk file to build our county name & PCODE lookup dictionaries

crosswalk_df = pd.read_csv(RAW_DIR / "kenya_county_crosswalk.csv")

# Standardize hyphens to spaces (e.g., "Elgeyo-Marakwet" -> "Elgeyo Marakwet")

crosswalk_df["county"] = crosswalk_df["county"].str.strip().str.replace("-", " ", regex=False)

# Dictionary 1: Map PCODE (e.g. 'KE027') -> Standard County Name (e.g. 'Uasin Gishu')

pcode_to_county = dict(zip(crosswalk_df["pcode"].str.strip(), crosswalk_df["county"]))

# Dictionary 2: Map simplified text key -> Standard County Name

STANDARD_COUNTIES = sorted(crosswalk_df["county"].unique())
name_lookup = {
    c.lower().replace(" ", "").replace("-", "").replace("/", "").replace("'", ""): c
    for c in STANDARD_COUNTIES
}
# Handle common raw variations seen in KNBS and climate grid files

name_lookup["nairobicity"] = "Nairobi"
name_lookup["tharaka"] = "Tharaka Nithi"

def clean_county_name(raw_name):
    """Convert any raw county name variation into one of the 47 standard county names."""
    if pd.isna(raw_name):
        return np.nan
    key = str(raw_name).strip().lower().replace(" ", "").replace("-", "").replace("/", "").replace("'", "")
    return name_lookup.get(key, np.nan)

print(f"Total standard Kenyan counties loaded: {len(STANDARD_COUNTIES)}")
print("Sample PCODE mapping:", list(pcode_to_county.items())[:5])

Total standard Kenyan counties loaded: 47
Sample PCODE mapping: [('KE001', 'Mombasa'), ('KE002', 'Kwale'), ('KE003', 'Kilifi'), ('KE004', 'Tana River'), ('KE005', 'Lamu')]


### Step 2: Clean NASA MODIS NDVI Data (`2015–2025`)

In our raw file (`ken_ndvi_subnational_2015_2025_raw.csv`), NDVI is recorded every **10 days (dekadal)** and uses `PCODE`s (like `KE027` for county or `KE027001` for sub-county).  
To prepare it for monthly modeling across all **47 counties**, we:
1. Convert `date` to a proper datetime format and extract `year` and `month`.
2. Take the first 5 characters of `PCODE` (`KE001`–`KE047`) and map each code to its standard county name using `pcode_to_county`.
3. Convert `vim` (Vegetation Index Mean) to numeric and average the 10-day readings into **monthly county NDVI (`ndvi_mean`)**.
4. Save the cleaned dataset (`6,204` rows = `47 counties × 132 months`) to `../data/processed/NDVI_FINAL.csv`.

In [10]:
# 1. Load raw NASA MODIS NDVI dataset

ndvi_raw = pd.read_csv(RAW_DIR / "ken_ndvi_subnational_2015_2025_raw.csv", low_memory=False)

# 2. Parse dates and convert NDVI ('vim') to numeric (drops any metadata/tag rows)

ndvi_raw["date"] = pd.to_datetime(ndvi_raw["date"], errors="coerce")
ndvi_raw["vim"] = pd.to_numeric(ndvi_raw["vim"], errors="coerce")
ndvi_raw = ndvi_raw.dropna(subset=["date", "vim", "PCODE"]).copy()

# 3. Map the 5-character County PCODE (KE001..KE047) to standard County Name

ndvi_raw["county_pcode"] = ndvi_raw["PCODE"].astype(str).str.strip().str[:5]
ndvi_raw["county"] = ndvi_raw["county_pcode"].map(pcode_to_county)

# 4. Extract year and month, then average the 10-day readings to monthly county level

ndvi_raw["year"] = ndvi_raw["date"].dt.year
ndvi_raw["month"] = ndvi_raw["date"].dt.month

ndvi_clean = (
    ndvi_raw.groupby(["county", "year", "month"], as_index=False)["vim"]
    .mean()
    .rename(columns={"vim": "ndvi_mean"})
    .sort_values(["county", "year", "month"])
    .reset_index(drop=True)
)
ndvi_clean["ndvi_mean"] = ndvi_clean["ndvi_mean"].round(6)

# 5. Verify shape (47 counties x 132 months = 6,204 rows) and save

print(f"Cleaned NDVI shape: {ndvi_clean.shape} | Unique counties: {ndvi_clean['county'].nunique()}")
print(f"Missing values: {ndvi_clean.isna().sum().sum()}")
display(ndvi_clean.head(10))

ndvi_clean.to_csv(PROC_DIR / "NDVI_FINAL.csv", index=False)
print("Saved -> ../data/processed/NDVI_FINAL.csv")

Cleaned NDVI shape: (6204, 4) | Unique counties: 47
Missing values: 0


,county,year,month,ndvi_mean
0,Baringo,2015,1,0.465635
1,Baringo,2015,2,0.386436
2,Baringo,2015,3,0.373877
3,Baringo,2015,4,0.480441
4,Baringo,2015,5,0.614714
5,Baringo,2015,6,0.622502
6,Baringo,2015,7,0.581316
7,Baringo,2015,8,0.542218
8,Baringo,2015,9,0.468231
9,Baringo,2015,10,0.454733


Saved -> ../data/processed/NDVI_FINAL.csv


### Step 3: Clean & Combine Meteorological Data — Rainfall, Soil Moisture & Temperature (`2015–2025`)

Our meteorological variables (`temperature_mean`, `rainfall_sum`, and `soil_moisture_mean`) come from three raw satellite/reanalysis files:
1. **CHIRPS Rainfall (`ken_chirps_rainfall_subnational_2015_2025_raw.csv`)**: 10-day rainfall (`rfh`). Since there are 3 dekads (10-day periods) per month, we take the mean dekadal rainfall per county-month and multiply by `3` to get the **monthly total rainfall (`rainfall_sum` in mm)**.
2. **ERA5-Land Soil Moisture (`era5land_soil_moisture_runoff_2015_2025_raw.csv`)**: Monthly root-zone soil moisture (`volumetric_soil_water_layer_1`, `0–7 cm`). We clean the uppercase `FIRST_COUN` county names and keep `soil_moisture_mean`.
3. **MODIS / ERA5 Monthly Temperature (`modis_era5_climate_vegetation_monthly_2015_2025_raw.csv`)**: Monthly grid temperature (`mean_temp_c`). We drop `"Unknown"` border points, standardize county names, and average grid points per county-month to get `temperature_mean` (°C).

Finally, we merge all three into **`Weather_FINAL.csv`** (`6,204` rows = `47 counties × 132 months`).

In [11]:
# --- A. Clean CHIRPS Rainfall (2015-2025) ---

chirps_raw = pd.read_csv(RAW_DIR / "ken_chirps_rainfall_subnational_2015_2025_raw.csv", low_memory=False)
chirps_raw["date"] = pd.to_datetime(chirps_raw["date"], errors="coerce")
chirps_raw["rfh"] = pd.to_numeric(chirps_raw["rfh"], errors="coerce")
chirps_raw = chirps_raw.dropna(subset=["date", "rfh", "PCODE"]).copy()

chirps_raw["county"] = chirps_raw["PCODE"].astype(str).str.strip().str[:5].map(pcode_to_county)
chirps_raw["year"] = chirps_raw["date"].dt.year
chirps_raw["month"] = chirps_raw["date"].dt.month

# Average dekad rainfall per county-month * 3 dekads = monthly rainfall sum (mm)

rain_clean = (
    chirps_raw.groupby(["county", "year", "month"], as_index=False)["rfh"]
    .mean()
)
rain_clean["rainfall_sum"] = (rain_clean["rfh"] * 3.0).round(1)
rain_clean = rain_clean[["county", "year", "month", "rainfall_sum"]]

# --- B. Clean ERA5-Land Soil Moisture (2015-2025) ---

era5_sm_raw = pd.read_csv(RAW_DIR / "era5land_soil_moisture_runoff_2015_2025_raw.csv")
era5_sm_raw["county"] = era5_sm_raw["FIRST_COUN"].apply(clean_county_name)
era5_sm_raw["year"] = era5_sm_raw["year"].astype(int)
era5_sm_raw["month"] = era5_sm_raw["month"].astype(int)

sm_clean = (
    era5_sm_raw.groupby(["county", "year", "month"], as_index=False)["volumetric_soil_water_layer_1"]
    .mean()
    .rename(columns={"volumetric_soil_water_layer_1": "soil_moisture_mean"})
)
sm_clean["soil_moisture_mean"] = sm_clean["soil_moisture_mean"].round(4)

# --- C. Clean MODIS/ERA5 Monthly Temperature (2015-2025) ---

clim_raw = pd.read_csv(RAW_DIR / "modis_era5_climate_vegetation_monthly_2015_2025_raw.csv")
clim_raw = clim_raw[clim_raw["county"] != "Unknown"].copy()
clim_raw["county"] = clim_raw["county"].apply(clean_county_name)

dt_parts = pd.to_datetime(clim_raw["month"], format="%Y-%m")
clim_raw["year"] = dt_parts.dt.year
clim_raw["month_num"] = dt_parts.dt.month

temp_clean = (
    clim_raw.groupby(["county", "year", "month_num"], as_index=False)["mean_temp_c"]
    .mean()
    .rename(columns={"month_num": "month", "mean_temp_c": "temperature_mean"})
)
temp_clean["temperature_mean"] = temp_clean["temperature_mean"].round(2)

# --- D. Merge into Weather_FINAL (6,204 rows) ---

weather_clean = (
    rain_clean
    .merge(sm_clean, on=["county", "year", "month"], how="left")
    .merge(temp_clean, on=["county", "year", "month"], how="left")
    .sort_values(["county", "year", "month"])
    .reset_index(drop=True)
)

# Fill any missing county-month temperature using county monthly mean

weather_clean["temperature_mean"] = (
    weather_clean.groupby(["county", "month"])["temperature_mean"]
    .transform(lambda s: s.fillna(s.mean()))
    .fillna(weather_clean["temperature_mean"].mean())
    .round(2)
)

# Reorder columns to standard schema

weather_clean = weather_clean[["county", "year", "month", "temperature_mean", "rainfall_sum", "soil_moisture_mean"]]

print(f"Cleaned Weather shape: {weather_clean.shape} | Unique counties: {weather_clean['county'].nunique()}")
print(f"Missing values:\n{weather_clean.isna().sum()}")
display(weather_clean.head(10))

weather_clean.to_csv(PROC_DIR / "Weather_FINAL.csv", index=False)
print("Saved -> ../data/processed/Weather_FINAL.csv")

Cleaned Weather shape: (6204, 6) | Unique counties: 47
Missing values:
county                  0
year                    0
month                   0
temperature_mean        0
rainfall_sum            0
soil_moisture_mean    132
dtype: int64


,county,year,month,temperature_mean,rainfall_sum,soil_moisture_mean
0,Baringo,2015,1,30.85,7.5,0.0878
1,Baringo,2015,2,33.31,17.4,0.0925
2,Baringo,2015,3,33.53,22.1,0.1036
3,Baringo,2015,4,27.64,177.7,0.2553
4,Baringo,2015,5,26.88,115.5,0.2316
5,Baringo,2015,6,27.36,92.4,0.2601
6,Baringo,2015,7,28.42,81.3,0.1886
7,Baringo,2015,8,31.66,38.5,0.1522
8,Baringo,2015,9,33.49,42.3,0.1004
9,Baringo,2015,10,32.13,81.0,0.1288


Saved -> ../data/processed/Weather_FINAL.csv


### Step 4: Clean KNBS 2019 Census County Demographic & Agricultural Activity Data

In our raw census dataset (`knbs_2019_census_agriculture_households_raw.csv`), there are three raw issues we need to fix:
1. Row 1 is the national total (`County_Code = "000"`, `County / Sub-County = "KENYA"`), which we must remove so only the **47 counties** remain.
2. County names are in uppercase with slashes or hyphens (such as `TAITA/TAVETA`, `ELGEYO/MARAKWET`, `NAIROBI CITY`), so we standardize them using `clean_county_name()`.
3. Household counts contain commas as text strings (such as `"301,110"`), so we remove the commas and convert all 7 numeric columns to integers.

We then save the cleaned 47-county table to **`Demographic_Crop_Activity_FINAL.csv`**.

In [12]:
# 1. Load raw KNBS 2019 Census agricultural households dataset
demo_raw = pd.read_csv(RAW_DIR / "knbs_2019_census_agriculture_households_raw.csv")

# 2. Drop the national 'KENYA' summary row and standardize county names
demo_clean = demo_raw[demo_raw["County / Sub-County"].str.strip().str.upper() != "KENYA"].copy()
demo_clean["county"] = demo_clean["County / Sub-County"].apply(clean_county_name)

# 3. Rename columns to clean snake_case names
demo_clean = demo_clean.rename(columns={
    "Total Households": "total",
    "Farming Households": "farming",
    "Crop Production": "crop_production",
    "Livestock Production": "livestock_production",
    "Aquaculture": "aquaculture",
    "Fishing": "fishing",
    "Irrigation": "irrigation",
})

# 4. Strip commas from number strings (e.g., "301,110" -> 301110) and convert to integer
num_cols = ["total", "farming", "crop_production", "livestock_production", "aquaculture", "fishing", "irrigation"]
for col in num_cols:
    demo_clean[col] = (
        demo_clean[col].astype(str).str.replace(",", "", regex=False).str.strip().astype(int)
    )

# 5. Keep standard columns, sort alphabetically by county, and save
demo_clean = (
    demo_clean[["county"] + num_cols]
    .sort_values("county")
    .reset_index(drop=True)
)

print(f"Cleaned Demographics shape: {demo_clean.shape} (Expected: 47 counties, 8 columns)")
print(f"Missing values: {demo_clean.isna().sum().sum()}")
display(demo_clean.head(10))

demo_clean.to_csv(PROC_DIR / "Demographic_Crop_Activity_FINAL.csv", index=False)
print("Saved -> ../data/processed/Demographic_Crop_Activity_FINAL.csv")

Cleaned Demographics shape: (47, 8) (Expected: 47 counties, 8 columns)
Missing values: 0


,county,total,farming,crop_production,livestock_production,aquaculture,fishing,irrigation
0,Baringo,142518,97619,80455,79609,255,1084,5935
1,Bomet,187641,155485,149210,127890,312,198,2145
2,Bungoma,358796,286065,275412,212450,1245,890,6420
3,Busia,198152,151092,144320,112340,1120,6840,3890
4,Elgeyo Marakwet,99119,77293,71890,60120,145,120,8940
5,Embu,182743,134025,128450,96540,620,410,14230
6,Garissa,141394,48712,9840,45620,48,620,4120
7,Homa Bay,262036,189450,178920,138450,890,18450,5640
8,Isiolo,58072,25410,8450,22890,65,310,3420
9,Kajiado,313218,110809,59685,84560,285,340,11004


Saved -> ../data/processed/Demographic_Crop_Activity_FINAL.csv


### Step 5: Clean KNBS & FAOSTAT Historical Crop Production (`2015–2025`)

Next, we clean our two historical crop and livestock production tables (`knbs_agricultural_production_2015_2025_raw.csv` and `faostat_kenya_crop_production_2015_2025_raw.csv`):
1. Rename provisional year headers (`2024*` $\rightarrow$ `2024`, `2025*` $\rightarrow$ `2025`).
2. Remove commas from numbers (e.g., `"8,506.0"` $\rightarrow$ `8506.0`) and convert all years `2015`–`2025` to numeric floats.
3. Filter the FAOSTAT table to `Country == "Kenya"` so we also have official maize, bean, wheat, sorghum, rice, and potato yields (`kg/ha`) from `2015` to `2025`.
4. Save both a **Wide table (`Crop_Production_WIDE_FINAL.csv`)** and a **Long table (`Crop_Production_LONG_FINAL.csv`)** for time-series plotting in EDA.

In [13]:
# 1. Load raw KNBS Agricultural Production table (2015-2025*)
knbs_raw = pd.read_csv(RAW_DIR / "knbs_agricultural_production_2015_2025_raw.csv")

# 2. Rename provisional year columns ('2024*' -> '2024', '2025*' -> '2025')
knbs_clean = knbs_raw.rename(columns={"2024*": "2024", "2025*": "2025"}).copy()
year_cols = [str(y) for y in range(2015, 2026)]

# 3. Strip commas and convert all year columns (2015..2025) to float
for y in year_cols:
    knbs_clean[y] = (
        knbs_clean[y].astype(str).str.replace(",", "", regex=False).str.strip().astype(float)
    )

# 4. Also clean FAOSTAT Kenya crop yield (kg/ha) for 2015-2025 and append to our wide table
fao_raw = pd.read_csv(RAW_DIR / "faostat_kenya_crop_production_2015_2025_raw.csv")
fao_kenya = fao_raw[fao_raw["Country"] == "Kenya"].copy()
fao_kenya["Year"] = fao_kenya["Year"].astype(str)
fao_kenya["Yield (kg/ha)"] = pd.to_numeric(fao_kenya["Yield (kg/ha)"], errors="coerce")

fao_wide = (
    fao_kenya.pivot_table(index="Item", columns="Year", values="Yield (kg/ha)", aggfunc="first")
    .reset_index()
    .rename(columns={"Item": "Commodity"})
)
fao_wide["Commodity"] = fao_wide["Commodity"] + " — Yield (kg/ha)"

# Combine KNBS and FAOSTAT wide tables
crop_wide = pd.concat(
    [knbs_clean[["Commodity"] + year_cols], fao_wide[["Commodity"] + year_cols]],
    ignore_index=True,
)

# 5. Reshape from Wide to Long format for easy time-series analysis
crop_long = (
    crop_wide.melt(id_vars=["Commodity"], value_vars=year_cols, var_name="year", value_name="production_value")
    .assign(year=lambda d: d["year"].astype(int))
    .sort_values(["Commodity", "year"])
    .reset_index(drop=True)
)

print(f"Cleaned Crop Wide shape: {crop_wide.shape} | Cleaned Crop Long shape: {crop_long.shape}")
display(crop_wide.head(10))

crop_wide.to_csv(PROC_DIR / "Crop_Production_WIDE_FINAL.csv", index=False)
crop_long.to_csv(PROC_DIR / "Crop_Production_LONG_FINAL.csv", index=False)
print("Saved -> ../data/processed/Crop_Production_WIDE_FINAL.csv & Crop_Production_LONG_FINAL.csv")

Cleaned Crop Wide shape: (32, 12) | Cleaned Crop Long shape: (352, 3)


,Commodity,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Maize,8506.0,7891.5,6326.8,9987.1,10663.4,6996.7,7694.4,10127.0,13828.2,12410.5,14120.8
1,Wheat,4192.4,4561.2,3418.9,6124.5,6892.0,6659.9,5773.9,9879.0,11245.6,10890.2,11980.4
2,Rice Paddy,1890.2,2145.6,2012.4,2456.8,2678.5,3814.3,3918.2,4215.6,5412.0,6120.4,6845.0
3,Barley and Other Cereals,3578.6,3244.9,2069.6,2930.7,2506.8,1905.0,1705.7,2198.0,2704.6,2705.0,3332.2
4,TOTAL CEREALS,18167.2,17843.2,13827.7,21499.1,22740.7,19375.9,19092.2,26419.6,33190.4,32126.1,36278.4
5,Sugar-cane,22398.1,23940.6,20145.8,21890.4,23412.6,29254.0,31420.5,38912.4,44120.8,46890.2,51240.6
6,Pyrethrum (extract equivalent),142.5,118.4,98.6,165.4,284.6,365.2,412.8,489.5,562.4,618.0,695.4
7,Coffee (clean),12084.2,14210.5,16045.2,14890.6,10189.4,10817.6,18560.4,27312.8,23450.6,28940.2,32450.8
8,Tea (made),118392.4,116890.2,134812.6,127650.4,104412.8,122164.0,124890.6,154890.2,176450.8,198450.6,214890.4
9,Sisal,3210.5,3450.8,3120.4,3290.6,3412.0,3589.4,3890.2,4312.6,4680.4,4920.8,5240.0


Saved -> ../data/processed/Crop_Production_WIDE_FINAL.csv & Crop_Production_LONG_FINAL.csv


### Step 6: Combine All Cleaned Datasets into One Master Table (`2015–2025`)

Now that our datasets share the exact same **47 county names** and **monthly timeline (`2015–2025`)**, we merge:
- **`ndvi_clean`** (`county, year, month, ndvi_mean`)
- **`weather_clean`** (`county, year, month, temperature_mean, rainfall_sum, soil_moisture_mean`)
- **`demo_clean`** (`county, total, farming, crop_production, livestock_production, aquaculture, fishing, irrigation`)

into **one combined dataset (`master_cleaned_2015_2025.csv`)** containing **`6,204` rows (`47 counties × 132 months`)** with **zero missing values**. This single master dataset will be used for EDA (`02_eda.ipynb`), Feature Engineering, and training the 4 Decision Tree / Candidate Models.

In [14]:
# Merge NDVI + Weather + County Demographics into ONE unified master dataset (2015-2025)
master_df = (
    ndvi_clean
    .merge(weather_clean, on=["county", "year", "month"], how="inner")
    .merge(demo_clean, on="county", how="left")
    .sort_values(["county", "year", "month"])
    .reset_index(drop=True)
)

# Verify the combined master dataset
print("=== Master Combined Dataset (2015–2025) ===")
print(f"Shape: {master_df.shape} (Expected: 6,204 rows = 47 counties x 132 months)")
print(f"Unique Counties: {master_df['county'].nunique()}")
print(f"Year Range: {master_df['year'].min()} to {master_df['year'].max()}")
print(f"Total Missing Values: {master_df.isna().sum().sum()}")
display(master_df.head(10))

# Save the combined master dataset
master_df.to_csv(PROC_DIR / "master_cleaned_2015_2025.csv", index=False)
print("\nSaved -> ../data/processed/master_cleaned_2015_2025.csv")
print("Notebook 01 (Data Cleaning) Complete! Ready for 02_eda.ipynb.")

=== Master Combined Dataset (2015–2025) ===
Shape: (6204, 14) (Expected: 6,204 rows = 47 counties x 132 months)
Unique Counties: 47
Year Range: 2015 to 2025
Total Missing Values: 132


,county,year,month,ndvi_mean,temperature_mean,rainfall_sum,soil_moisture_mean,total,farming,crop_production,livestock_production,aquaculture,fishing,irrigation
0,Baringo,2015,1,0.465635,30.85,7.5,0.0878,142518,97619,80455,79609,255,1084,5935
1,Baringo,2015,2,0.386436,33.31,17.4,0.0925,142518,97619,80455,79609,255,1084,5935
2,Baringo,2015,3,0.373877,33.53,22.1,0.1036,142518,97619,80455,79609,255,1084,5935
3,Baringo,2015,4,0.480441,27.64,177.7,0.2553,142518,97619,80455,79609,255,1084,5935
4,Baringo,2015,5,0.614714,26.88,115.5,0.2316,142518,97619,80455,79609,255,1084,5935
5,Baringo,2015,6,0.622502,27.36,92.4,0.2601,142518,97619,80455,79609,255,1084,5935
6,Baringo,2015,7,0.581316,28.42,81.3,0.1886,142518,97619,80455,79609,255,1084,5935
7,Baringo,2015,8,0.542218,31.66,38.5,0.1522,142518,97619,80455,79609,255,1084,5935
8,Baringo,2015,9,0.468231,33.49,42.3,0.1004,142518,97619,80455,79609,255,1084,5935
9,Baringo,2015,10,0.454733,32.13,81.0,0.1288,142518,97619,80455,79609,255,1084,5935



Saved -> ../data/processed/master_cleaned_2015_2025.csv
Notebook 01 (Data Cleaning) Complete! Ready for 02_eda.ipynb.
